# Control Evaluator

> Fill in a module description here

In [ ]:
#| default_exp evaluators.control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange
import hydra
from c3jepa_wm.utils import channel

In [ ]:
#| export
class MultiAgentGoalEvaluator:
    """
    Dataset-driven evaluation of the JEPA planner for a 2-agent communicative setting.

    For each episode and each agent:
      - pick a start step `t0` (e.g. random or fixed)
      - the agent's goal is its own observation at `t0 + goal_offset`
      - the *other* agent encodes its own current observation via the VQ-VAE
        into `msg_indices` and "sends" it (used as conditioning for this agent's rollout)
      - run the planner to get a sequence of actions
      - compare against ground-truth actions / measure goal-reaching error

    Args:
        model: JEPA model (has .encoder, .predictor, .action_encoder, .get_cost, .rollout)
        vqvae: sender-side VQ-VAE module; vqvae.encode(pixels) -> (B, 1, 49) token indices
        planner_cls: planner class (e.g. JEPAGoalPlanner)
        planner_kwargs: kwargs to construct one planner per agent
        history_size: number of past steps fed as context (matches JEPA.rollout's history_size)
        goal_offset: number of steps ahead the goal is set
        agents: list of agent keys, must have length 2
        device: torch device
    """

    def __init__(
        self,
        data_module,
        model,
        planner_cfg,
        slurm_jobid= None,
        history_size=3,
        goal_offset=10,
        agents=("agent_0", "agent_1"),
        device="cpu",
        SNR = 10.0,  # example SNR for power/schedule extraction; in practice, this could be tuned or provided as part of the episode
        p_max = 10,  # max power level for scheduling decisions
        noise_power=1e-3,  # noise power for SNR calculations
    ):
        assert len(agents) == 2, "This evaluator only supports exactly 2 agents."
        self.data_module = data_module
        self.data_module.setup()

        self.model = model['jepa'].to(device).eval()
        self.vqvae = model['vqvae'].to(device).eval()
        self.agents = list(agents)
        self.history_size = history_size
        self.goal_offset = goal_offset
        self.device = device
        self.SNR = SNR
        self.p_max = p_max
        self.noise_power = noise_power

        # one planner per agent (mirrors the per-agent action_dim/horizon if they differ)
        self.planners = {
            # agent: planner_cls(model=self.model, device=device, **planner_kwargs)
            agent: hydra.utils.instantiate(planner_cfg, model= self.model, device=device)
            for agent in self.agents
        }

    @torch.no_grad()
    def _encode_message(self, partner_pixels_vqvae_t0, csi_t0, schedule=None, power=None, no_comm=False):
        """
        partner_pixels_vqvae_t0: (B, C, H, W) -- partner's obs at t0, VQ-VAE-transform space
        csi_t0: (1,) complex -- channel state at t0 for this sender->receiver link
        schedule, power: scalars or (1,) tensors, or None to skip channel entirely
        """
        partner_pixels_vqvae_t0 = partner_pixels_vqvae_t0.to(self.device)
        indices = self.vqvae.get_message_indices(partner_pixels_vqvae_t0)  # (B, 1, H, W)
        indices = rearrange(indices, "B H W -> B (H W)").long() # (B, 49)

        if schedule is not None and power is not None:
            n = 1  # single neighbor (the other agent)
            print(csi_t0.shape, schedule.shape, power.shape)
            csi = csi_t0 #csi_t0.reshape(1, n, 1).to(self.device)              # (B*T=1, n, 1) complex
            schedule = torch.as_tensor(schedule, device=self.device)#.reshape(1, n, 1)
            power = torch.as_tensor(power, device=self.device)#.reshape(1, n, 1)

            indices = channel(
                schedule=schedule,
                power=power,
                msg_indices=indices,       # (B*T, 49) -> here B*T = 1
                csi=csi,
                device=self.device,
                snr_db=self.SNR,
                no_comm=no_comm,
            )  # returns (B*T, 49) = (1, 49)

        return indices.unsqueeze(1)  # (1, 1, 49) -- matches msg_indices shape expected downstream
        
    def _extract_power_and_schedule(self, csi):
        snr_linear = 10 ** (self.SNR / 10.0)
        optimal_power = snr_linear * self.noise_power / (torch.abs(csi) ** 2 + 1e-8)
        schedule = (optimal_power <= self.p_max).float()
        return optimal_power, schedule
        
    def _build_agent_info(self, episode, agent, partner, t0):
        H = self.history_size
        pixels = episode[agent]["pixels"][:, t0 - H + 1 : t0 + 1]
        actions = episode[agent]["action"][:, t0 - H + 1 : t0 + 1]
        goal_pixels = episode[agent]["pixels"][:, t0 + self.goal_offset]

        partner_obs_vqvae_t0 = episode[partner]["pov_seq_vqvae"][:, t0]#.unsqueeze(0)
        csi_t0 = episode[partner]["csi"][:, t0]#.unsqueeze(0)  # (1,) complex -- confirm this is the right link's CSI
        power_level, schedule = self._extract_power_and_schedule(csi_t0)
        msg_indices = self._encode_message(
            partner_obs_vqvae_t0, csi_t0, schedule= schedule, power= power_level
        )

        info = {
            "pixels": pixels.unsqueeze(0).to(self.device),
            "action": actions.unsqueeze(0).to(self.device),
            "goal": goal_pixels.unsqueeze(0).to(self.device),
            "msg_indices": msg_indices.to(self.device),
        }
        return info


    @torch.no_grad()
    def evaluate_episode(self, episode, t0=None):
        """
        episode: dict keyed by agent -> {"pixels": (T,C,H,W), "action": (T,action_dim)}
        t0: start step; if None, picked so that t0 - H + 1 >= 0 and t0 + goal_offset < T
        Returns per-agent dict with planned actions, ground-truth actions, and goal error.
        """
        H = self.history_size
        T = episode[self.agents[0]]["pixels"].size(1)
        # print(episode[self.agents[0]]["pixels"].shape)
        lo = H - 1
        hi = T - self.goal_offset - 1
        print(H, T, lo, hi)
        assert hi >= lo, "Episode too short for given history_size/goal_offset."
        if t0 is None:
            t0 = torch.randint(lo, hi + 1, (1,)).item()

        results = {}
        for agent in self.agents:
            partner = [a for a in self.agents if a != agent][0]
            info = self._build_agent_info(episode, agent, partner, t0)

            first_action, plan = self.planners[agent].plan(info)

            gt_future_actions = episode[agent]["action"][t0 + 1 : t0 + 1 + plan.size(1)]
            goal_err = self.planners[agent].eval_plan(info, self.device, plan) #self._goal_error(episode, agent, t0, plan)

            results[agent] = {
                "t0": t0,
                "planned_actions": plan.squeeze(0).cpu(),
                "first_action": first_action.cpu(),
                "gt_actions": gt_future_actions.cpu(),
                "goal_error": goal_err,
            }
        return results

    @torch.no_grad()
    def _goal_error(self, episode, agent, t0, plan):
        """Roll the *true* dynamics forward isn't available (no env here), so this
        measures embedding-space distance between the model's own final predicted
        state and the true goal embedding -- i.e. how well the model itself thinks
        the plan reaches the goal. Swap for env-based execution if you have a sim.
        """
        H = self.history_size
        pixels = episode[agent]["pixels"][t0 - H + 1 : t0 + 1].unsqueeze(0).to(self.device)
        actions = episode[agent]["action"][t0 - H + 1 : t0 + 1].unsqueeze(0).to(self.device)
        goal_pixels = episode[agent]["pixels"][t0 + self.goal_offset].unsqueeze(0).to(self.device)

        info = {"pixels": pixels, "action": actions, "goal": goal_pixels}
        action_seq = torch.cat([actions, plan.to(self.device)], dim=1).unsqueeze(1)  # add S=1 dim
        info_for_cost = {k: (v.unsqueeze(1) if torch.is_tensor(v) else v) for k, v in info.items()}
        cost = self.model.get_cost(info_for_cost, action_seq)  # (1, 1)
        return cost.item()

    @torch.no_grad()
    def evaluate_dataset(self, num_episodes=None):
        """
        dataset: iterable of episodes (each a dict as in evaluate_episode)
        Returns a list of per-episode results plus aggregate stats.
        """
        dataset = self.data_module.val_dataloader()
        all_results = []
        for i, episode in enumerate(dataset):
            if num_episodes is not None and i >= num_episodes:
                break
            all_results.append(self.evaluate_episode(episode))

        agg = {agent: {"goal_error": []} for agent in self.agents}
        for ep_res in all_results:
            for agent in self.agents:
                agg[agent]["goal_error"].append(ep_res[agent]["goal_error"])

        summary = {
            agent: {
                "mean_goal_error": float(torch.tensor(agg[agent]["goal_error"]).mean()),
                "std_goal_error": float(torch.tensor(agg[agent]["goal_error"]).std()),
                "n": len(agg[agent]["goal_error"]),
            }
            for agent in self.agents
        }

        return {"per_episode": all_results, "summary": summary}
    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()